# Phase 7 — Recommendation System

Notebook 07. A content-based recommender (TF-IDF + cosine similarity over
product name + description) via `src/recommendation.py`, plus an optional
Sentence-Transformer embedding-based recommender, and a simple offline
evaluation (category-precision@k).

> **Environment note:** `sentence-transformers` isn't installed in this
> sandbox (and downloading model weights needs network access), so only the
> TF-IDF recommender runs below. `build_embedding_recommender()` will pick up
> `sentence-transformers` automatically once installed — no code changes
> needed.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
%matplotlib inline


In [ ]:
from src.eda import load_features
from src.recommendation import TfidfRecommender, category_precision_at_k

df = load_features()
recommender = TfidfRecommender(df)


## Example: Recommend Similar Products

In [ ]:
query = "Wayona Nylon Braided USB"
recommender.recommend(product_name=query, top_k=5)


**Result:** all 5 recommendations are USB/charging cables from the same or a
closely related product family, with similarity scores > 0.7 — the TF-IDF
recommender is correctly capturing product similarity from name + description
text alone.

## Offline Evaluation: Category-Precision@5

In [ ]:
precision = category_precision_at_k(recommender, k=5, n_samples=200)
precision


**Interpretation:** ~94% of the top-5 recommendations share the queried
product's `main_category`. This is a reasonable proxy for topical relevance
given the lack of explicit relevance labels, though it will systematically
undervalue *cross-category* recommendations that are nonetheless a good fit
(e.g. a phone case recommended alongside a phone).

## Embedding-Based Recommender (Sentence-Transformers)

In [ ]:
from src.recommendation import build_embedding_recommender

embed_recommender = build_embedding_recommender(df)
if embed_recommender is not None:
    display(embed_recommender.recommend(product_id=df["product_id"].iloc[0], top_k=5))
else:
    print("sentence-transformers not available in this environment — TF-IDF recommender remains the working default.")
